# 04 — DenseNet121 5-Fold OOF Baseline

Mục tiêu: xác nhận baseline DenseNet121 trên toàn bộ development set bằng 5-fold CV, tạo OOF predictions và chọn threshold trên OOF. Internal holdout vẫn **niêm phong**.


In [ ]:
from pathlib import Path
import json, time, random, gc
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score

SEED=2026
PROJECT_ROOT=Path(r'D:\medicine')
MANIFEST_PATH=PROJECT_ROOT/'split_outputs'/'split_manifest.csv'
OUTPUT_DIR=PROJECT_ROOT/'cv_outputs'/'densenet121_224'
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
IMAGE_SIZE=224; BATCH_SIZE=16; EPOCHS=8; PATIENCE=3
LEARNING_RATE=1e-4; WEIGHT_DECAY=1e-4; NUM_WORKERS=0
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP=DEVICE.type=='cuda'
print('PyTorch:',torch.__version__)
print('Device:',DEVICE)
if USE_AMP: print('GPU:',torch.cuda.get_device_name(0))
print('Manifest exists:',MANIFEST_PATH.exists())


In [ ]:
manifest=pd.read_csv(MANIFEST_PATH)
dev=manifest[manifest['split']=='dev'].copy().reset_index(drop=True)
holdout=manifest[manifest['split']=='holdout'].copy()
assert len(dev)==4362 and len(holdout)==770
assert set(dev['sha256']).isdisjoint(set(holdout['sha256']))
print('Dev:',len(dev),dev['label'].value_counts().to_dict())
print('SEALED holdout:',len(holdout),holdout['label'].value_counts().to_dict())


In [ ]:
class PadToSquare:
    def __call__(self,img):
        w,h=img.size; side=max(w,h)
        l=(side-w)//2; r=side-w-l; t=(side-h)//2; b=side-h-t
        return ImageOps.expand(img,border=(l,t,r,b),fill=0)

MEAN=[0.485,0.456,0.406]; STD=[0.229,0.224,0.225]
train_tf=transforms.Compose([
    PadToSquare(),transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),
    transforms.RandomAffine(degrees=5,translate=(0.02,0.02),scale=(0.95,1.05),fill=0),
    transforms.ColorJitter(brightness=0.08,contrast=0.08),
    transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
val_tf=transforms.Compose([PadToSquare(),transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),transforms.ToTensor(),transforms.Normalize(MEAN,STD)])

LABEL_TO_INT={'Normal':0,'Pneumonia':1}
class XrayDataset(Dataset):
    def __init__(self,frame,tf): self.frame=frame.reset_index(drop=True); self.tf=tf
    def __len__(self): return len(self.frame)
    def __getitem__(self,i):
        row=self.frame.iloc[i]
        with Image.open(row['path']) as im: im=im.convert('RGB')
        return self.tf(im),LABEL_TO_INT[row['label']],row['filename']


In [ ]:
def metrics(y_true,probs,threshold=0.5):
    y_true=np.asarray(y_true,dtype=int); probs=np.asarray(probs,float)
    pred=(probs>=threshold).astype(int)
    tn,fp,fn,tp=confusion_matrix(y_true,pred,labels=[0,1]).ravel()
    acc=(tp+tn)/len(y_true); se=tp/max(tp+fn,1); sp=tn/max(tn+fp,1)
    f1=f1_score(y_true,pred,pos_label=1,zero_division=0)
    P=.30*acc+.25*se+.20*sp+.25*f1
    auc=roc_auc_score(y_true,probs)
    return {'P':float(P),'Accuracy':float(acc),'Sensitivity':float(se),'Specificity':float(sp),'F1':float(f1),'AUROC':float(auc),'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp),'threshold':float(threshold)}

@torch.no_grad()
def predict(model,loader):
    model.eval(); ys=[]; ps=[]; names=[]
    for x,y,n in loader:
        x=x.to(DEVICE,non_blocking=True)
        with torch.amp.autocast(device_type='cuda',enabled=USE_AMP):
            logits=model(x)
        p=torch.softmax(logits.float(),1)[:,1]
        ys.extend(y.numpy()); ps.extend(p.cpu().numpy()); names.extend(list(n))
    return np.asarray(ys),np.asarray(ps),names

def make_model():
    m=models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
    m.classifier=nn.Linear(m.classifier.in_features,2)
    return m.to(DEVICE)


## Train 5 folds

In [ ]:
all_oof=[]; fold_summaries=[]

for fold in range(5):
    print('\n'+'='*70); print('FOLD',fold); print('='*70)
    random.seed(SEED+fold); np.random.seed(SEED+fold); torch.manual_seed(SEED+fold)
    if USE_AMP: torch.cuda.manual_seed_all(SEED+fold)
    train_df=dev[dev['cv_fold']!=fold].copy().reset_index(drop=True)
    val_df=dev[dev['cv_fold']==fold].copy().reset_index(drop=True)
    assert set(train_df['sha256']).isdisjoint(set(val_df['sha256']))

    train_loader=DataLoader(XrayDataset(train_df,train_tf),batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=USE_AMP)
    val_loader=DataLoader(XrayDataset(val_df,val_tf),batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=USE_AMP)
    model=make_model()
    c=train_df['label'].value_counts(); n0=int(c['Normal']); n1=int(c['Pneumonia']); total=n0+n1
    cw=torch.tensor([total/(2*n0),total/(2*n1)],dtype=torch.float32,device=DEVICE)
    criterion=nn.CrossEntropyLoss(weight=cw)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=EPOCHS)
    scaler=torch.amp.GradScaler('cuda',enabled=USE_AMP)

    best_score=-1; best_epoch=0; best_state=None; stale=0; hist=[]
    for epoch in range(1,EPOCHS+1):
        t0=time.time(); model.train(); loss_sum=0; seen=0
        for x,y,_ in train_loader:
            x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type='cuda',enabled=USE_AMP):
                logits=model(x); loss=criterion(logits,y)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            loss_sum+=float(loss.item())*len(x); seen+=len(x)
        scheduler.step()
        yv,pv,_=predict(model,val_loader); m=metrics(yv,pv,0.5); train_loss=loss_sum/seen
        hist.append({'epoch':epoch,'train_loss':train_loss,**m})
        print(f"Epoch {epoch:02d} | loss={train_loss:.4f} | P={m['P']:.5f} | A={m['Accuracy']:.5f} | Se={m['Sensitivity']:.5f} | Sp={m['Specificity']:.5f} | F1={m['F1']:.5f} | {time.time()-t0:.0f}s")
        if m['P']>best_score+1e-6:
            best_score=m['P']; best_epoch=epoch; stale=0
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else:
            stale+=1
            if stale>=PATIENCE: print('Early stopping'); break

    model.load_state_dict(best_state)
    yv,pv,names=predict(model,val_loader)
    fold_m=metrics(yv,pv,0.5)
    pred_df=pd.DataFrame({'filename':names,'y_true':yv,'p_pneumonia':pv,'fold':fold})
    pred_df.to_csv(OUTPUT_DIR/f'fold{fold}_predictions.csv',index=False)
    pd.DataFrame(hist).to_csv(OUTPUT_DIR/f'fold{fold}_history.csv',index=False)
    torch.save({'architecture':'densenet121','fold':fold,'image_size':IMAGE_SIZE,'best_epoch':best_epoch,'state_dict':best_state},OUTPUT_DIR/f'fold{fold}_best.pt')
    fold_summaries.append({'fold':fold,'best_epoch':best_epoch,**fold_m})
    all_oof.append(pred_df)
    print('Fold metrics:',fold_m)
    del model,optimizer,scheduler,scaler,best_state,train_loader,val_loader
    gc.collect()
    if USE_AMP: torch.cuda.empty_cache()


## OOF score and robust threshold

In [ ]:
oof=pd.concat(all_oof,ignore_index=True)
assert len(oof)==len(dev)==4362
assert oof['filename'].nunique()==len(oof)
oof.to_csv(OUTPUT_DIR/'oof_predictions.csv',index=False)

fold_table=pd.DataFrame(fold_summaries)
display(fold_table)
print('Fold P mean:',fold_table['P'].mean(),'std:',fold_table['P'].std(ddof=1))

base=metrics(oof.y_true,oof.p_pneumonia,0.5)
thresholds=np.round(np.arange(0.05,0.951,0.005),3)
scan=pd.DataFrame([metrics(oof.y_true,oof.p_pneumonia,t) for t in thresholds])
best=scan.sort_values(['P','threshold'],ascending=[False,True]).iloc[0]
best_t=float(best.threshold); tuned=metrics(oof.y_true,oof.p_pneumonia,best_t)
scan.to_csv(OUTPUT_DIR/'oof_threshold_scan.csv',index=False)

summary={
  'model':'DenseNet121','pretrained':'ImageNet DEFAULT','image_size':IMAGE_SIZE,
  'fold_P_mean':float(fold_table['P'].mean()),'fold_P_std':float(fold_table['P'].std(ddof=1)),
  'oof_metrics_at_0_5':base,'oof_best_threshold':best_t,'oof_tuned_metrics':tuned,
  'holdout_used':False
}
with open(OUTPUT_DIR/'oof_summary.json','w',encoding='utf-8') as f: json.dump(summary,f,indent=2)
print(json.dumps(summary,indent=2))


## Interpretation rule

- Chỉ dùng OOF để chọn architecture / augmentation / loss / resolution / threshold.
- Không đánh giá internal holdout cho đến khi pipeline được chốt.
- Model tiếp theo chỉ đáng giữ nếu cải thiện OOF score đủ ổn định, không chỉ thắng một fold.
